# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset follows the [Croissant schema](https://mlcommons.org/croissant/) and is provided at:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

This notebook references all data entities (record sets, fields, etc.) by their `@id` as per best practice.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print high-level metadata
print(f"Dataset name: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")

## 2. Data Overview
Review available record sets, their fields, and associated `@id`s. The `mlcroissant` Dataset object makes this accessible. We'll print the record set `@id`s, their field `@id`s, and provide a small sample from each.

In [ ]:
from pprint import pprint

# List all record sets by @id
print('Available Record Sets:')
record_set_objs = getattr(metadata, 'record_set', [])

# If only a single record set, convert to list
if isinstance(record_set_objs, (dict, mlc.schemas.RecordSet)):
    record_set_objs = [record_set_objs]

record_set_ids = []

for rs in record_set_objs:
    rs_id = getattr(rs, '@id', None)
    if not rs_id:
        rs_id = getattr(rs, 'id', None)
    rs_name = getattr(rs, 'name', 'N/A')
    print(f"- Record Set @id: {rs_id}")
    print(f"  Name: {rs_name}")
    record_set_ids.append(rs_id)
    # List the fields
    fields = getattr(rs, 'field', [])
    # Some recordsets expose .field as single or list
    if isinstance(fields, (dict, mlc.schemas.Field)):
        fields = [fields]
    print("  Field @ids:")
    for f in fields:
        f_id = getattr(f, '@id', getattr(f, 'id', ''))
        f_name = getattr(f, 'name', 'N/A')
        f_data_type = getattr(f, 'data_type', 'N/A')
        print(f"    - {f_id} (name: {f_name}, type: {f_data_type})")
    # Try to show one record sample
    try:
        record_iter = dataset.records(record_set=rs_id) if rs_id else None
        if record_iter:
            example = next(record_iter)
            print("  Example Record:")
            pprint(example)
    except Exception as e:
        print(f"  No sample available or cannot load records: {e}")
    print()

## 3. Data Extraction
Load all records from each record set into a pandas DataFrame for analysis. All references use the record set and field `@id`s from the overview above.

In [ ]:
# Prepare all DataFrames for all available record sets
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records for record set: {rs_id}")
    try:
        records_iter = dataset.records(record_set=rs_id)
        records_list = list(records_iter)
        df = pd.DataFrame(records_list)
        dataframes[rs_id] = df
        print(f"Loaded {df.shape[0]} records, columns: {list(df.columns)}\n")
    except Exception as e:
        print(f"Failed to load record set {rs_id}: {e}\n")

# Print available DataFrame keys
print(f"Extracted DataFrames for record sets: {list(dataframes.keys())}")
# Show head of the first DataFrame (if any loaded)
example_rs_id = record_set_ids[0] if record_set_ids else None
if example_rs_id:
    display_cols = dataframes[example_rs_id].columns.tolist()
    print(f"Columns of DataFrame for {example_rs_id}: {display_cols}")
    display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Perform basic EDA: filtering, transforming, grouping, and normalizing. You can modify the `numeric_field_id` and `group_field_id` depending on fields found above.

In [ ]:
# Example field selection for this dataset:
# These should be updated according to the fields in the loaded DataFrame for your chosen record set.
# You may need to update `example_rs_id`, `numeric_field_id`, and `group_field_id` as per your exploration above.

if record_set_ids:
    df = dataframes[example_rs_id]
    print(f"Columns available: {df.columns.tolist()}")
    # Try to auto-select a numeric field (float or int)
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]  # The first numeric field
        print(f"Using numeric field: {numeric_field_id}")
    else:
        numeric_field_id = None
        print('No numeric field found for EDA.')

    # Try a categorical/grouping field (pick a likely string field that's not an id)
    candidate_group_cols = [col for col in df.columns if col.lower() in ['group', 'sample', 'type', 'description'] and col != numeric_field_id]
    group_field_id = candidate_group_cols[0] if candidate_group_cols else None
    print(f"Using group field: {group_field_id}")

    # (Below: EDA steps only if numeric field present)
    if numeric_field_id is not None:
        # Filtering: e.g., keep records with values > 10 in the numeric column
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold} (kept {len(filtered_df)} records)")

        # Normalization (z-score)
        if len(filtered_df) > 0:
            filtered_df[f"{numeric_field_id}_normalized"] = (
                filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
            ) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id}:")
            print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Grouping by a field (if present)
            if group_field_id and group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
                print(grouped_df.head())
else:
    print("No record set found for EDA.")

## 5. Visualization
Visualize distributions of a numeric field, and (optionally) a grouped aggregation, using `matplotlib`. Update fields as needed based on exploration.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for normalized numeric field if available
if record_set_ids and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group (if available)
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(
            data=filtered_df,
            x=group_field_id,
            y=numeric_field_id
        )
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've loaded the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using its Croissant schema, explored metadata and record structures, extracted tabular data, and performed basic data analysis and visualization using field and record set `@id`s. 

Key next steps might include more advanced statistics, machine learning, or domain-specific visualizations.